# Neuroselect Step 11 on Google Colab

This notebook runs the locked 3,990-span held-out language research evaluation with the four already-trained adapters. It does **not** train adapters and does not insert intended targets. GPU inference, the active checkpoint, and result assembly use Colab's local SSD; exact source, model cache, atomic checkpoint mirrors, and final artifacts remain durable on Google Drive.

Before running: select a T4 GPU runtime, upload `step11-language-inputs-v1.tar.gz` and `neuroselect-step11-source.bundle` to the Drive path configured below, and paste the bundle's exact 40-character Git SHA into `GIT_REVISION`.

In [ ]:
# Fail before installing anything if Colab assigned an incompatible GPU.
import platform
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

import torch

assert platform.machine() == "x86_64", platform.machine()
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime > Change runtime type > GPU"
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
memory_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}; compute capability {capability[0]}.{capability[1]}; {memory_gib:.1f} GiB")
assert capability >= (7, 5), (
    "This GPU is too old for MLX-CUDA. Reconnect until Colab assigns T4, L4, A100, "
    "or another NVIDIA GPU with compute capability >= 7.5."
)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# EDIT GIT_REVISION. Keep the other defaults unless your Drive layout differs.
GIT_REVISION = "PASTE_THE_40_CHARACTER_COMMIT_SHA_HERE"
DRIVE_ROOT = Path("/content/drive/MyDrive/neuroselect-step11")
SOURCE_BUNDLE_PATH = DRIVE_ROOT / "neuroselect-step11-source.bundle"
BUNDLE_PATH = DRIVE_ROOT / "step11-language-inputs-v1.tar.gz"
DRIVE_CHECKPOINT_DIR = DRIVE_ROOT / "checkpoint-optimized-v1"
DRIVE_RESULT_DIR = DRIVE_ROOT / "held-out-language-personalization-research-v1"
DRIVE_HF_HOME = DRIVE_ROOT / "huggingface-cache"
LOCAL_CHECKPOINT_DIR = Path("/content/neuroselect-step11-checkpoint")
LOCAL_RESULT_DIR = Path("/content/held-out-language-personalization-research-v1")
LOCAL_HF_HOME = Path("/content/neuroselect-huggingface-cache")
REPOSITORY_DIR = Path("/content/neuroselect-bci-step11")

assert len(GIT_REVISION) == 40 and all(c in "0123456789abcdef" for c in GIT_REVISION), (
    "Paste the exact 40-character Git commit SHA containing the Colab implementation."
)
assert SOURCE_BUNDLE_PATH.is_file(), f"Upload the exact source bundle to {SOURCE_BUNDLE_PATH}"
assert BUNDLE_PATH.is_file(), f"Upload the verified input bundle to {BUNDLE_PATH}"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_HF_HOME.mkdir(parents=True, exist_ok=True)
print(f"Drive workspace: {DRIVE_ROOT}")

In [ ]:
# Start from the uploaded private-repository bundle at a detached, byte-exact commit.
if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
bundle_heads = subprocess.run(
    ["git", "bundle", "list-heads", str(SOURCE_BUNDLE_PATH)],
    check=True,
    capture_output=True,
    text=True,
).stdout.splitlines()
assert any(line.startswith(GIT_REVISION + " ") for line in bundle_heads), (
    "The uploaded source bundle does not contain GIT_REVISION. Recreate and re-upload it."
)
subprocess.run(["git", "clone", str(SOURCE_BUNDLE_PATH), str(REPOSITORY_DIR)], check=True)
subprocess.run(["git", "checkout", "--detach", GIT_REVISION], cwd=REPOSITORY_DIR, check=True)
resolved_revision = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPOSITORY_DIR, check=True, capture_output=True, text=True
).stdout.strip()
status = subprocess.run(
    ["git", "status", "--porcelain"], cwd=REPOSITORY_DIR, check=True, capture_output=True, text=True
).stdout
assert resolved_revision == GIT_REVISION and not status
print(f"Clean checkout: {resolved_revision}")

In [ ]:
# Install the locked Python 3.12 environment and MLX CUDA 12 backend.
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.10.6"], check=True)
subprocess.run(["uv", "python", "install", "3.12"], cwd=REPOSITORY_DIR, check=True)
subprocess.run(
    ["uv", "sync", "--extra", "local-language-cuda", "--no-dev", "--locked", "--python", "3.12"],
    cwd=REPOSITORY_DIR,
    check=True,
)
PYTHON = REPOSITORY_DIR / ".venv/bin/python"
RUN_ENV = {
    **dict(__import__("os").environ),
    "HF_HOME": str(LOCAL_HF_HOME),
    "MPLBACKEND": "Agg",
    "PYTHONUNBUFFERED": "1",
    "TOKENIZERS_PARALLELISM": "false",
}
subprocess.run(
    [str(PYTHON), "scripts/check_language_cuda.py"], cwd=REPOSITORY_DIR, env=RUN_ENV, check=True
)

In [ ]:
# Verify/extract inputs, persist the pinned Qwen snapshot once, then copy it to local SSD.
subprocess.run(
    [
        str(PYTHON),
        "scripts/manage_language_cloud_bundle.py",
        "extract",
        str(BUNDLE_PATH),
        "--destination",
        str(REPOSITORY_DIR),
    ],
    cwd=REPOSITORY_DIR,
    env=RUN_ENV,
    check=True,
)
subprocess.run(
    [str(PYTHON), "scripts/cache_language_model.py", "--download"],
    cwd=REPOSITORY_DIR,
    env={**RUN_ENV, "HF_HOME": str(DRIVE_HF_HOME)},
    check=True,
)
if LOCAL_HF_HOME.exists():
    shutil.rmtree(LOCAL_HF_HOME)
shutil.copytree(DRIVE_HF_HOME, LOCAL_HF_HOME)
subprocess.run(
    [str(PYTHON), "scripts/cache_language_model.py"],
    cwd=REPOSITORY_DIR,
    env=RUN_ENV,
    check=True,
)
print(f"Pinned model cache copied to local SSD: {LOCAL_HF_HOME}")

## Short pilot

Run this before the full job. It uses the development message limit but the same pinned model and all four research adapters. It confirms inference, adapter switching, memory use, and approximate per-span speed. Its output is not research evidence.

In [ ]:
PILOT_DIR = Path("/content/neuroselect-step11-pilot")
_pilot = subprocess.run(
    [
        str(PYTHON),
        "scripts/run_held_out_language_evaluation.py",
        "--config",
        "configs/experiments/held_out_language_personalization.yaml",
        "--adapter-suffix=-research-v1",
        "--output",
        str(PILOT_DIR),
        "--progress-every",
        "1",
        "--overwrite",
    ],
    cwd=REPOSITORY_DIR,
    env=RUN_ENV,
    check=True,
)
print("Optimized pilot completed. Read the final projected full Step 11 duration above.")

## Full Step 11

This evaluates all 3,990 spans. Inference and per-trial writes stay on local SSD; every 25 newly completed spans are fsynced locally and atomically mirrored to Google Drive. Re-run the notebook after a disconnect: `--resume` restores the exact mirror and skips every valid checkpointed span.

In [ ]:
_full = subprocess.run(
    [
        str(PYTHON),
        "scripts/run_held_out_language_evaluation.py",
        "--config",
        "configs/experiments/held_out_language_personalization_research.yaml",
        "--adapter-suffix=-research-v1",
        "--output",
        str(LOCAL_RESULT_DIR),
        "--checkpoint-dir",
        str(LOCAL_CHECKPOINT_DIR),
        "--checkpoint-mirror-dir",
        str(DRIVE_CHECKPOINT_DIR),
        "--resume",
        "--checkpoint-every",
        "25",
        "--progress-every",
        "10",
        "--overwrite",
    ],
    cwd=REPOSITORY_DIR,
    env=RUN_ENV,
    check=True,
)
print("Full Step 11 evaluation completed locally; run the verification/export cell next.")

## Strict verification and export

The verifier checks the research protocol, all 3,990 ordered teacher-forced spans, four local adapter/corpus checksums, candidate vocabulary, clean producing commit, run identity, artifact checksums, and claim eligibility.

In [ ]:
subprocess.run(
    [
        str(PYTHON),
        "scripts/verify_language_research_evaluation.py",
        "--artifacts",
        str(LOCAL_RESULT_DIR),
    ],
    cwd=REPOSITORY_DIR,
    env=RUN_ENV,
    check=True,
)
if DRIVE_RESULT_DIR.exists():
    shutil.rmtree(DRIVE_RESULT_DIR)
shutil.copytree(LOCAL_RESULT_DIR, DRIVE_RESULT_DIR)
EXPORT_PATH = DRIVE_ROOT / "held-out-language-personalization-research-v1.tar.gz"
with tarfile.open(EXPORT_PATH, "w:gz") as archive:
    archive.add(LOCAL_RESULT_DIR, arcname=LOCAL_RESULT_DIR.name)
print(f"Verified Step 11 export: {EXPORT_PATH}")